# 🔷 Módulo 10 - Notebook 03: Mapas de densidad y optimización de sucursales

## 🏪 Visualización y decisión estratégica con H3

**Libro:** Saliendo de lo Pandito  
**Módulo:** 10 - Indexación Hexagonal Uber H3  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Crear** mapas de densidad hexagonales  
✅ **Visualizar** hot-spots de actividad comercial  
✅ **Identificar** zonas sin cobertura  
✅ **Optimizar** ubicaciones de sucursales  
✅ **Detectar** canibalización entre sucursales

---

## 📋 Pre-requisitos

* ✅ Notebooks 10_01 y 10_02 completados
* ✅ Conocimiento de agregación H3
* ✅ Familiaridad con visualizaciones geoespaciales

---

## 📚 Contenido

1. Mapas de Densidad Hexagonales
2. Identificación de Hot-Spots
3. Análisis de Cobertura
4. Detección de Canibalización
5. Optimización de Ubicaciones
6. Caso Integrador: Estrategia de Expansión

---

## 💡 Por qué importa

**H3 transforma decisiones estratégicas:**

* 🏪 **Expansión:** ¿Dónde abrir la próxima sucursal?
* 📍 **Cobertura:** ¿Qué zonas no atendemos?
* ⚠️ **Canibalización:** ¿Sucursales muy cercanas?
* 📊 **Performance:** ¿Dónde están las mejores ventas?

**De datos a decisiones en minutos**

In [0]:
import pandas as pd
import numpy as np

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (con coordenadas)
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
    
    # Extraer ubicaciones de sucursales
    df_sucursales = df_ventas[['sucursal_id', 'sucursal_nombre', 'zona', 'lat', 'lon']].drop_duplicates().reset_index(drop=True)
    
    # Agregar ventas totales por sucursal
    ventas_totales = df_ventas.groupby('sucursal_id')['ventas'].sum().reset_index()
    ventas_totales.columns = ['sucursal_id', 'ventas_totales']
    df_sucursales = df_sucursales.merge(ventas_totales, on='sucursal_id')
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros ventas: {len(df_ventas):,}")
    print(f"   🏪 Sucursales: {len(df_sucursales)}")
    print(f"   📅 Período: {df_ventas['fecha'].min().strftime('%Y-%m-%d')} a {df_ventas['fecha'].max().strftime('%Y-%m-%d')}")
    print(f"   📍 Ubicación: Mendoza, Argentina")
    
    print(f"\n💰 Métricas de negocio:")
    print(f"   • Ventas totales: ${df_ventas['ventas'].sum():,.2f}")
    print(f"   • Sucursal top: {df_sucursales.loc[df_sucursales['ventas_totales'].idxmax(), 'sucursal_nombre']}")
    print(f"   • Ventas top: ${df_sucursales['ventas_totales'].max():,.2f}")
    
    print(f"\n🎯 Este notebook creará:")
    print(f"   • Mapas de densidad hexagonales")
    print(f"   • Análisis de cobertura territorial")
    print(f"   • Recomendaciones de optimización")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_ventas = None
    df_sucursales = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 Mapas de Densidad y Optimización con H3

### 🗺️ Mapas de Densidad Hexagonales

**Concepto:** Agregar métricas de negocio por hexágono y visualizar con escala de color.

**Proceso:**

```python
import h3

# 1. Convertir a H3
df['h3'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)

# 2. Agregar ventas por hexágono
hex_ventas = df.groupby('h3')['ventas'].sum().reset_index()

# 3. Visualizar
# Hexágonos rojos = alta densidad
# Hexágonos azules = baja densidad
```

---

### 🔥 Identificación de Hot-Spots

**Hot-Spot:** Hexágono con actividad significativamente mayor al promedio.

**Método:**

```python
# Calcular percentiles
p75 = hex_ventas['ventas'].quantile(0.75)
p90 = hex_ventas['ventas'].quantile(0.90)

# Clasificar
hex_ventas['categoria'] = 'Normal'
hex_ventas.loc[hex_ventas['ventas'] >= p75, 'categoria'] = '🟡 Alto'
hex_ventas.loc[hex_ventas['ventas'] >= p90, 'categoria'] = '🔴 Hot-Spot'
```

**Uso:** Priorizar inversiones en zonas hot-spot.

---

### 📍 Análisis de Cobertura

**Pregunta:** ¿Qué zonas NO tienen sucursales cercanas?

**Método:**

1. **Crear buffers H3 alrededor de sucursales**
   ```python
   # Obtener hexágonos vecinos (radio 2 = ~2km)
   for sucursal_hex in sucursales['h3']:
       vecinos = h3.k_ring(sucursal_hex, 2)
   ```

2. **Identificar hexágonos sin cobertura**
   ```python
   # Hexágonos con actividad pero sin sucursal cercana
   sin_cobertura = hex_ventas[~hex_ventas['h3'].isin(vecinos_totales)]
   ```

**Resultado:** Zonas de oportunidad para expansión.

---

### ⚠️ Detección de Canibalización

**Canibalización:** Dos sucursales muy cercanas compitiendo por los mismos clientes.

**Método:**

```python
import h3

# Calcular distancia entre sucursales (en hexágonos)
for i, suc_a in sucursales.iterrows():
    for j, suc_b in sucursales.iterrows():
        if i < j:
            distancia = h3.h3_distance(suc_a['h3'], suc_b['h3'])
            if distancia <= 3:  # Menos de 3 hexágonos (~300m)
                print(f"⚠️  Canibalización: {suc_a['nombre']} y {suc_b['nombre']}")
```

**Acción:** Cerrar una sucursal o reasignar zonas.

---

### 🎯 Optimización de Ubicaciones

**Pregunta:** ¿Dónde abrir la próxima sucursal?

**Criterios:**

1. **Alta densidad de ventas** (hot-spot)
2. **Sin cobertura actual** (no hay sucursal cercana)
3. **Alta población** (datos externos)

**Proceso:**

```python
# 1. Identificar hexágonos hot-spot
hot_spots = hex_ventas[hex_ventas['categoria'] == '🔴 Hot-Spot']

# 2. Filtrar hexágonos sin cobertura
sin_cobertura = hot_spots[~hot_spots['h3'].isin(vecinos_totales)]

# 3. Ordenar por ventas
sin_cobertura = sin_cobertura.sort_values('ventas', ascending=False)

# 4. Top candidato
top_candidato = sin_cobertura.iloc[0]
print(f"Abrir sucursal en hex: {top_candidato['h3']}")
print(f"Ventas potenciales: ${top_candidato['ventas']:,.2f}")
```

---

### 📊 Visualización de Estrategia

**Mapa final:**

* 🔴 **Hexágonos rojos:** Hot-spots actuales
* 🟢 **Hexágonos verdes:** Sucursales existentes
* ⭐ **Estrella amarilla:** Ubicación propuesta nueva sucursal
* ⚪ **Hexágonos grises:** Zonas sin actividad

---

### 💼 Caso de Uso Empresarial Completo

**Contexto:** Cadena de supermercados con 10 sucursales en Mendoza.

**Pregunta:** ¿Dónde abrir la sucursal #11?

**Análisis:**

1. Agregar ventas por hex (res 9)
2. Identificar hot-spots (top 10%)
3. Filtrar hexágonos sin cobertura (>2km de sucursales)
4. Cruzar con datos de población
5. Seleccionar top candidato

**Resultado:** Hexágono `89a8100c54fffff` con:
* Ventas potenciales: $2.5M/año
* Población: 15,000 habitantes
* Distancia a sucursal más cercana: 3.2km

**Decisión:** Abrir sucursal en ese hexágono.

---

### ⚡ Ventajas de H3 para Optimización
✅ **Velocidad:** Análisis de millones de transacciones en segundos  
✅ **Precisión:** Geometría uniforme (sin sesgos de cuadrículas)  
✅ **Escalabilidad:** De ciudad a continente  
✅ **Visualización:** Mapas ejecutivos instantáneos

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🗺️ MAPAS DE DENSIDAD Y OPTIMIZACIÓN H3")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    import h3
    print(f"Versión de H3: {h3.__version__}")
except ImportError:
    print("⚠️  H3 no instalado. Ejecuta: %pip install h3")

print("\n🎯 En este notebook aprenderás:")
print("  • Crear mapas de densidad hexagonales")
print("  • Identificar hot-spots comerciales")
print("  • Detectar zonas sin cobertura")
print("  • Optimizar ubicaciones de sucursales")

print("\n📖 Métodos clave:")
print("  - h3.k_ring(hex_id, k)  # Hexágonos vecinos")
print("  - h3.h3_distance(hex_a, hex_b)  # Distancia en hex")
print("  - df.groupby('h3')['ventas'].sum()  # Densidad")
print("  - df['ventas'].quantile(0.90)  # Hot-spots")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

## 🗺️ Mapas de densidad y optimización con H3 y datos reales

### 🔥 Hot-spots comerciales

Con los datos de **Los Andes Market** podemos identificar hexágonos con ventas anormalmente altas usando percentiles:

```python
# Agregar ventas por hexágono (res 8)
densidad = df.groupby('h3_res8')['ventas'].sum().reset_index()

# Clasificar por percentiles
p75 = densidad['ventas'].quantile(0.75)
p90 = densidad['ventas'].quantile(0.90)

# Hot-spot: ventas > p90
```

---

### 📍 Análisis de cobertura con k_ring

```python
# Hexágonos cubiertos por cada sucursal (radio 2 = ~1km)
cobertura = set()
for hex_id in sucursales['h3_res8']:
    cobertura.update(h3.grid_disk(hex_id, 2))

# Hexágonos sin cobertura = todos - cobertura
```

---

### ⚠️ Canibalización

Dos sucursales en el mismo hexágono (o en hexágonos adyacentes) compiten por los mismos clientes.

```python
# Sucursales que comparten hexágono res 8
dupes = df.groupby('h3_res8')['sucursal_id'].nunique()
canibalizadas = dupes[dupes > 1]
```

---

### 💡 Preguntas estratégicas
* ¿Dónde están los hot-spots de ventas?
* ¿Qué zonas no tienen cobertura (gaps)?
* ¿Hay sucursales que se canibalizan entre sí?
* ¿Dónde abrir la próxima sucursal?

In [0]:
import pandas as pd
import h3
import plotly.express as px
import numpy as np

print("🗺️ MAPAS DE DENSIDAD Y OPTIMIZACIÓN CON DATOS REALES")
print("="*70)

if USAR_DATOS_REALES and df_ventas is not None:
    print("\n1️⃣  MAPA DE DENSIDAD: Ventas por hexágono (res 8)")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        densidad = df_ventas.groupby('h3_res8').agg(
            ventas_total=('ventas', 'sum'),
            num_sucursales=('sucursal_id', 'nunique'),
            zona=('zona', 'first')
        ).reset_index()

        # Calcular centro de cada hexágono
        densidad['lat'] = densidad['h3_res8'].apply(lambda x: h3.h3_to_geo(x)[0])
        densidad['lon'] = densidad['h3_res8'].apply(lambda x: h3.h3_to_geo(x)[1])
        densidad['ventas_por_suc'] = densidad['ventas_total'] / densidad['num_sucursales']

        fig = px.scatter(
            densidad, x='lon', y='lat',
            size='ventas_total', color='ventas_total',
            hover_name='h3_res8', hover_data=['num_sucursales', 'zona'],
            title='🗺️ Densidad de Ventas por Hexágono H3 (Res 8)',
            labels={'ventas_total': 'Ventas ($)', 'lat': 'Lat', 'lon': 'Lon'},
            template='plotly_white', size_max=40,
            color_continuous_scale='YlOrRd'
        )
        fig.show()

    print("\n" + "="*70)
    print("\n2️⃣  HOT-SPOTS: Identificación por percentiles")
    print("-"*70)

    p50 = densidad['ventas_total'].quantile(0.50)
    p75 = densidad['ventas_total'].quantile(0.75)
    p90 = densidad['ventas_total'].quantile(0.90)

    densidad['categoria'] = 'Normal'
    densidad.loc[densidad['ventas_total'] >= p75, 'categoria'] = 'Alto'
    densidad.loc[densidad['ventas_total'] >= p90, 'categoria'] = 'Hot-Spot'

    print(f"\n   Clasificación de hexágonos:")
    print(densidad['categoria'].value_counts())
    print(f"\n   Percentiles: p50=${p50:,.0f}  p75=${p75:,.0f}  p90=${p90:,.0f}")

    fig_hot = px.scatter(
        densidad, x='lon', y='lat',
        size='ventas_total', color='categoria',
        hover_name='h3_res8',
        title='🔥 Hot-Spots Comerciales - Los Andes Market',
        template='plotly_white', size_max=40,
        color_discrete_map={'Normal': '#60a5fa', 'Alto': '#f59e0b', 'Hot-Spot': '#ef4444'}
    )
    fig_hot.show()

    print("\n" + "="*70)
    print("\n3️⃣  CANIBALIZACIÓN: Sucursales en el mismo hexágono")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        sucursales_por_hex = df_ventas.groupby('h3_res8')['sucursal_id'].nunique()
        hex_con_duplicados = sucursales_por_hex[sucursales_por_hex > 1]

        print(f"\n   Hexágonos con múltiples sucursales (res 8): {len(hex_con_duplicados)}")
        if len(hex_con_duplicados) > 0:
            for hex_id in hex_con_duplicados.index:
                sucursales_hex = df_ventas[df_ventas['h3_res8'] == hex_id]['sucursal_nombre'].unique()
                ventas_hex = df_ventas[df_ventas['h3_res8'] == hex_id].groupby('sucursal_nombre')['ventas'].sum()
                print(f"\n   Hexágono {hex_id}:")
                print(f"      Sucursales: {list(sucursales_hex)}")
                print(f"      Ventas por sucursal: {ventas_hex.round(0).to_dict()}")
            print("\n   ⚠️  Canibalización potencial - sucursales muy cercanas")
        else:
            print("\n   ✅ No hay sucursales que compartan hexágono (res 8)")

    print("\n" + "="*70)
    print("\n4️⃣  COBERTURA: Hexágonos vecinos por sucursal")
    print("-"*70)

    if 'h3_res8' in df_ventas.columns:
        # Cobertura con radio k=2 (~920m en res 8)
        cobertura_total = set()
        hex_sucursales = df_ventas['h3_res8'].unique()

        for hex_id in hex_sucursales:
            vecinos = h3.grid_disk(hex_id, 2)
            cobertura_total.update(vecinos)

        print(f"\n   Hexágonos con cobertura (k=2): {len(cobertura_total)}")
        print(f"   Hexágonos de sucursales: {len(hex_sucursales)}")
        print(f"   Ratio cobertura/sucursales: {len(cobertura_total) / len(hex_sucursales):.1f}x")
        print("\n   💡 Cada sucursal cubre ~19 hexágonos vecinos (k=2)")

    print("\n" + "="*70)
    print("\n5️⃣  RANKING DE DENSIDAD POR ZONA")
    print("-"*70)

    ranking_zona = densidad.groupby('zona').agg(
        hexágonos=('h3_res8', 'count'),
        ventas_total=('ventas_total', 'sum'),
        sucursales=('num_sucursales', 'sum')
    ).sort_values('ventas_total', ascending=False)

    ranking_zona['ventas_por_hex'] = ranking_zona['ventas_total'] / ranking_zona['hexágonos']
    ranking_zona['ventas_por_suc'] = ranking_zona['ventas_total'] / ranking_zona['sucursales']

    print("\n   Ranking de zonas por densidad comercial:")
    print(ranking_zona.round(0))
    print("\n   💡 Mayor ventas_por_suc = mayor productividad por sucursal")
else:
    print("⚠️  No hay datos reales disponibles")

print("\n" + "="*70)

In [0]:
# 🎯 OPCIONAL: Mapas de densidad y clustering con datos reales

# Descomentar para usar datos reales:
"""
import pandas as pd
import h3

print("💾 Cargando datos con H3 desde Unity Catalog...")

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    df_ventas = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3").toPandas()
    
    print(f"✅ Datos cargados: {len(df_ventas):,} registros")
    
    # Calcular densidad de ventas por hexágono (resolución 8)
    densidad_ventas = df_ventas.groupby('h3_res8').agg({
        'ventas': 'sum',
        'sucursal_id': 'nunique'
    }).reset_index()
    densidad_ventas.columns = ['h3_id', 'ventas_totales', 'num_sucursales']
    densidad_ventas['ventas_por_sucursal'] = densidad_ventas['ventas_totales'] / densidad_ventas['num_sucursales']
    
    print(f"\n🗺️ Densidad de Ventas por Hexágono (res 8):")
    print(f"   Hexágonos únicos: {len(densidad_ventas)}")
    print(f"   Ventas totales: ${densidad_ventas['ventas_totales'].sum():,.2f}")
    display(densidad_ventas.sort_values('ventas_totales', ascending=False).head())
    
    # Identificar clusters de sucursales (hexágonos con múltiples sucursales)
    clusters = densidad_ventas[densidad_ventas['num_sucursales'] > 1].copy()
    
    print(f"\n🎯 Clusters Identificados:")
    if len(clusters) > 0:
        print(f"   Hexágonos con múltiples sucursales: {len(clusters)}")
        for idx, row in clusters.iterrows():
            print(f"   • {row['h3_id']}: {row['num_sucursales']} sucursales")
    else:
        print("   No hay clusters (todas las sucursales están en hexágonos diferentes)")
    
    # Análisis de cobertura territorial
    print(f"\n📍 Análisis de Cobertura Territorial:")
    
    # Obtener todas las sucursales con sus vecinos
    df_sucursales = df_ventas[[
        'sucursal_id', 'sucursal_nombre', 'h3_res8', 'zona'
    ]].drop_duplicates()
    
    for idx, suc in df_sucursales.iterrows():
        h3_id = suc['h3_res8']
        vecinos = h3.grid_disk(h3_id, 1)  # Hex + vecinos inmediatos
        
        # Contar cuantas sucursales hay en vecinos
        sucursales_vecinas = df_sucursales[
            df_sucursales['h3_res8'].isin(vecinos)
        ]['sucursal_id'].nunique() - 1  # -1 para excluir la sucursal misma
        
        print(f"   {suc['sucursal_id']}: {sucursales_vecinas} sucursales vecinas")
    
    # Mapa de calor conceptual
    print(f"\n🔥 Hexágonos con Mayor Densidad de Ventas:")
    top_hexagons = densidad_ventas.nlargest(5, 'ventas_totales')
    for idx, row in top_hexagons.iterrows():
        # Obtener centro del hexágono
        lat, lon = h3.cell_to_latlng(row['h3_id'])
        print(f"   • {row['h3_id']}")
        print(f"      Ubicación: ({lat:.4f}, {lon:.4f})")
        print(f"      Ventas: ${row['ventas_totales']:,.2f}")
        print(f"      Sucursales: {row['num_sucursales']}")
    
    print(f"\n💡 Variables disponibles:")
    print("   • densidad_ventas: DataFrame con densidad por hexágono")
    print("   • clusters: Hexágonos con múltiples sucursales")
    print("   • df_sucursales: Sucursales únicas con sus H3")
    
    print(f"\n🗺️ Ideas para Visualización:")
    print("   1. Mapa de calor con Folium (heatmap)")
    print("   2. Hexágonos coloreados por densidad de ventas")
    print("   3. Marcadores de sucursales con popup de información")
    print("   4. Isolineas de zonas de influencia")
    print("   5. Clustering espacial con DBSCAN o KMeans")
    
    print(f"\n🎯 Recomendaciones de Optimización:")
    if len(clusters) > 0:
        print("   ⚠️  Hay clusters de sucursales cercanas - evaluar cannibalización")
    else:
        print("   ✅ Buena distribución espacial - baja competencia interna")
    
    # Identificar gaps (zonas sin cobertura)
    print(f"\n🔍 Oportunidades de Expansión:")
    print("   Usar H3 para identificar hexágonos sin cobertura cerca de zonas de alta densidad")
    
except Exception as e:
    print(f"⚠️  Tabla no encontrada: {e}")
    print("   Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
"""

print("ℹ️  Usando datos sintéticos de este notebook")
print("="*70)

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')

## 🎓 Conclusiones del Notebook 10_03

### ✅ Lo que aprendiste

1. **Mapas de densidad hexagonales:**
   ```python
   # Agregar ventas por hexágono y visualizar densidad
   df['h3'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)
   hex_ventas = df.groupby('h3')['ventas'].sum().reset_index()
   # Hexágonos rojos = alta densidad, azules = baja densidad
   ```

2. **Identificación de hot-spots:**
   ```python
   # Clasificar por percentiles
   p75 = hex_ventas['ventas'].quantile(0.75)
   p90 = hex_ventas['ventas'].quantile(0.90)
   hex_ventas['categoria'] = 'Normal'
   hex_ventas.loc[hex_ventas['ventas'] >= p75, 'categoria'] = '🟡 Alto'
   hex_ventas.loc[hex_ventas['ventas'] >= p90, 'categoria'] = '🔴 Hot-Spot'
   ```

3. **Análisis de cobertura territorial:**
   ```python
   # Hexágonos vecinos de cada sucursal (radio k=2 ≈ 2km)
   for sucursal_hex in sucursales['h3']:
       vecinos = h3.grid_disk(sucursal_hex, 2)
   # Identificar zonas con actividad pero sin sucursal cercana
   sin_cobertura = hex_ventas[~hex_ventas['h3'].isin(vecinos_totales)]
   ```

4. **Detección de canibalización:**
   ```python
   # Distancia entre sucursales en hexágonos
   for i, suc_a in sucursales.iterrows():
       for j, suc_b in sucursales.iterrows():
           if i < j:
               distancia = h3.grid_distance(suc_a['h3'], suc_b['h3'])
               if distancia <= 3:  # Menos de 3 hexágonos
                   print(f"⚠️  Canibalización: {suc_a['nombre']} ↔ {suc_b['nombre']}")
   ```

5. **Optimización de ubicaciones:**
   - Intersectar hot-spots con zonas sin cobertura
   - Ordenar candidatos por ventas potenciales
   - Seleccionar top candidato para nueva sucursal

---

### 🛠️ Guía rápida de optimización con H3

**Caso 1: Mapa de densidad de ventas**
```python
df['h3'] = df.apply(lambda r: h3.geo_to_h3(r['lat'], r['lon'], 9), axis=1)
densidad = df.groupby('h3')['ventas'].sum().reset_index()
densidad = densidad.sort_values('ventas', ascending=False)
```

**Caso 2: Clasificar hot-spots por percentiles**
```python
p90 = densidad['ventas'].quantile(0.90)
hot_spots = densidad[densidad['ventas'] >= p90]
print(f"🔥 {len(hot_spots)} hot-spots identificados (top 10%)")
```

**Caso 3: Cobertura territorial con grid_disk**
```python
radio = 2  # hexágonos de radio (~2km a res 8)
vecinos_totales = set()
for h in sucursales['h3']:
    vecinos_totales.update(h3.grid_disk(h, radio))
# Zonas sin cobertura: hot-spots fuera del radio
oportunidades = hot_spots[~hot_spots['h3'].isin(vecinos_totales)]
```

**Caso 4: Detectar canibalización entre sucursales**
```python
for i in range(len(sucursales)):
    for j in range(i+1, len(sucursales)):
        dist = h3.grid_distance(sucursales.iloc[i]['h3'], sucursales.iloc[j]['h3'])
        if dist <= 3:
            print(f"⚠️  {sucursales.iloc[i]['sucursal_nombre']} ↔ {sucursales.iloc[j]['sucursal_nombre']}: {dist} hex")
```

**Caso 5: Recomendar nueva ubicación**
```python
candidatos = hot_spots[~hot_spots['h3'].isin(vecinos_totales)]
top = candidatos.sort_values('ventas', ascending=False).iloc[0]
lat, lon = h3.cell_to_latlng(top['h3'])
print(f"⭐ Nueva sucursal en ({lat:.4f}, {lon:.4f})")
print(f"   Ventas potenciales: ${top['ventas']:,.2f}")
```

---

### 🏆 Resumen del Módulo 10

**Aprendiste:**

1. **10_01 - Indexación Hexagonal Uber H3:** Sistema H3, resoluciones, conversión geo↔hex
2. **10_02 - Agregación Espacial y Resoluciones:** Agregación por hex, jerarquía padre-hijo, multiescala
3. **10_03 - Mapas de Densidad y Optimización:** Hot-spots, cobertura, canibalización, expansión

**Habilidades adquiridas:**
* ✅ Crear mapas de densidad hexagonales de actividad comercial
* ✅ Identificar hot-spots con percentiles y clasificación automática
* ✅ Analizar cobertura territorial y detectar zonas sin atención
* ✅ Detectar canibalización entre sucursales cercanas
* ✅ Recomendar ubicaciones óptimas para nuevas sucursales

---


<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🔷 ¡Módulo 10 Completado!</h3>
  <p><i>"Dominas H3: desde indexación hexagonal hasta optimización estratégica de sucursales. Ahora puedes tomar decisiones de expansión basadas en datos espaciales."</i></p>
</div>